# NovaSuite —  Exploratory Data Analysis (EDA)

---


In [ ]:
import pandas as pd
import numpy as np

# ── Load raw tables ──────────────────────────────────────────────────────────
customers = pd.read_csv("../project/customers.csv")
plans     = pd.read_csv("../project/plans.csv")
revenue   = pd.read_csv("../project/revenue.csv")
usage     = pd.read_csv("../project/usage.csv")
churn     = pd.read_csv("../project/churn.csv")

# ── Datetime conversions ─────────────────────────────────────────────────────
customers["signup_date"] = pd.to_datetime(customers["signup_date"])
revenue["month"]         = pd.to_datetime(revenue["month"])
usage["month"]           = pd.to_datetime(usage["month"])
churn["churn_date"]      = pd.to_datetime(churn["churn_date"])

# ── String standardization ───────────────────────────────────────────────────
for col in ["industry", "company_size", "country"]:
    customers[col] = customers[col].str.strip().str.title()

# ── Build master dataframe ───────────────────────────────────────────────────
# Step 1: Merge revenue with usage on customer + month
master = pd.merge(revenue, usage, on=["customer_id", "month"], how="inner")

# Step 2: Merge with customers to bring in firmographic fields
master = pd.merge(master, customers[["customer_id", "company_name", "industry",
                                      "company_size", "signup_date", "country"]],
                  on="customer_id", how="left")

# Step 3: Merge with plans to bring in plan name and pricing metadata
master = pd.merge(master, plans[["plan_id", "plan_name", "monthly_price", "max_users"]],
                  on="plan_id", how="left")

# Step 4: Attach churn flag
churned_ids = set(churn["customer_id"])
master["is_churned"] = master["customer_id"].isin(churned_ids).astype(int)

print("Master dataframe built successfully.")
print(f"Shape: {master.shape}")

---
## 1. Master Dataframe Overview



In [ ]:
# ── 1a. Shape ────────────────────────────────────────────────────────────────
rows, cols = master.shape
print(f"Rows : {rows:,}")
print(f"Columns : {cols}")

In [ ]:
# ── 1b. Column names ─────────────────────────────────────────────────────────
print("Columns:")
for i, col in enumerate(master.columns, 1):
    print(f"  {i:>2}. {col}")

In [ ]:
# ── 1c. Data types and null counts ───────────────────────────────────────────
print(master.info())

In [ ]:
# ── 1d. First five rows ──────────────────────────────────────────────────────
master.head()

**Interpretation:**  
Each row represents one customer's activity in one calendar month. The master table joins all five source files and is the single analytical surface for all subsequent analysis. Null counts should be zero (or near-zero) for all key fields after the cleaning step.

---
## 2. Customer Composition



In [ ]:
# ── 2a. Industry distribution ────────────────────────────────────────────────
# Work at the customer level (one row per customer)
customer_profile = customers.copy()

industry_counts = customer_profile["industry"].value_counts()
industry_pct    = (industry_counts / len(customer_profile) * 100).round(1)

industry_summary = pd.DataFrame({
    "Customer Count": industry_counts,
    "Share (%)": industry_pct
})

print("=== Industry Distribution ===")
print(industry_summary.to_string())

In [ ]:
# ── 2b. Company size distribution ────────────────────────────────────────────
size_order  = ["Small", "Medium", "Large", "Enterprise"]
size_counts = customer_profile["company_size"].value_counts().reindex(size_order, fill_value=0)
size_pct    = (size_counts / len(customer_profile) * 100).round(1)

size_summary = pd.DataFrame({
    "Customer Count": size_counts,
    "Share (%)": size_pct
})

print("=== Company Size Distribution ===")
print(size_summary.to_string())

In [ ]:
# ── 2c. Country distribution ─────────────────────────────────────────────────
country_counts = customer_profile["country"].value_counts()
country_pct    = (country_counts / len(customer_profile) * 100).round(1)

country_summary = pd.DataFrame({
    "Customer Count": country_counts,
    "Share (%)": country_pct
})

print("=== Country Distribution ===")
print(country_summary.to_string())

**Interpretation:**  
- **Industry:** Identifies which segments dominate the customer base. EdTech and FinTech are flagged in the problem statement as high-priority segments, so their relative weight matters.  
- **Company Size:** A heavy concentration of Small companies is expected; the key business question is whether they are on appropriately priced plans.  
- **Country:** USA and India are the primary markets. Geographic concentration informs pricing and support strategy.

---
## 3. Plan Adoption

Examining how the 500 customers are distributed across the four subscription tiers.  
Plan mix directly determines the MRR ceiling and signals whether customers are on appropriately sized plans.

**Plans:**
| Plan | List Price | Max Users |
|------|-----------|-----------|
| Basic | \$29/mo | 5 |
| Pro | \$79/mo | 20 |
| Business | \$149/mo | 50 |
| Enterprise | \$299/mo | 200 |

In [ ]:
# ── 3a. Customers by plan ────────────────────────────────────────────────────
plan_counts = customer_profile["plan_id"].value_counts()

# Map plan IDs to names for readability
plan_id_to_name = plans.set_index("plan_id")["plan_name"].to_dict()
plan_counts.index = plan_counts.index.map(plan_id_to_name)

# Ensure logical order
plan_order  = ["Basic", "Pro", "Business", "Enterprise"]
plan_counts = plan_counts.reindex(plan_order, fill_value=0)
plan_pct    = (plan_counts / plan_counts.sum() * 100).round(1)

plan_summary = pd.DataFrame({
    "Customer Count": plan_counts,
    "Share (%)": plan_pct
})

print("=== Plan Adoption ===")
print(plan_summary.to_string())

In [ ]:
# ── 3b. Revenue-weighted plan view (using list price × customer count) ────────
plan_info = plans.set_index("plan_name")[["monthly_price", "max_users"]]

plan_summary["List Price (\$)"]    = plan_info["monthly_price"].values
plan_summary["Max Users"]          = plan_info["max_users"].values
plan_summary["Theoretical MRR (\$)"] = (
    plan_summary["Customer Count"] * plan_summary["List Price (\$)"]
)

print("=== Plan Adoption with Theoretical MRR ===")
print(plan_summary.to_string())
print(f"\nTotal Theoretical MRR (no discounts): \${plan_summary['Theoretical MRR (\$)'].sum():,.0f}")

**Interpretation:**  
- Plans with high customer counts but low list prices (e.g., Basic) may limit MRR growth potential.  
- A large share of Enterprise or Business customers signals upsell success — or over-provisioning risk if company sizes are small.  
- Theoretical MRR vs actual MRR (calculated in the Revenue Analysis notebook) will quantify the discount leakage.

---
## 4. Revenue Overview



In [ ]:
# ── 4a. Descriptive stats for revenue fields ─────────────────────────────────
revenue_cols = ["base_revenue", "subscription_revenue", "expansion_revenue", "discount"]

rev_stats = master[revenue_cols].describe().round(2)
print("=== Revenue Field Statistics (all months, all customers) ===")
print(rev_stats.to_string())

In [ ]:
# ── 4b. Proportion of months with non-zero expansion revenue ─────────────────
total_records    = len(master)
expansion_months = (master["expansion_revenue"] > 0).sum()
expansion_pct    = round(expansion_months / total_records * 100, 1)

print(f"Total monthly records      : {total_records:,}")
print(f"Months with expansion rev  : {expansion_months:,}")
print(f"Expansion revenue rate     : {expansion_pct}% of all customer-months")

In [ ]:
# ── 4c. Proportion of months where a discount was applied ────────────────────
discount_months = (master["discount"] > 0).sum()
discount_pct    = round(discount_months / total_records * 100, 1)

print(f"Months with discount applied : {discount_months:,}")
print(f"Discount rate                : {discount_pct}% of all customer-months")

**Interpretation:**  
- The gap between `base_revenue` mean and `subscription_revenue` mean represents average discount leakage per customer-month.  
- A low expansion revenue rate (% of months with `expansion_revenue > 0`) confirms the weak upsell motion flagged in the problem statement.  
- High standard deviation in `expansion_revenue` suggests a small number of customers drive the bulk of expansion dollars.

---
## 5. Usage Overview



In [ ]:
# ── 5a. Descriptive stats for usage fields ───────────────────────────────────
usage_cols = ["active_users", "sessions", "login_count", "feature_usage"]

usage_stats = master[usage_cols].describe().round(2)
print("=== Usage Field Statistics (all months, all customers) ===")
print(usage_stats.to_string())

In [ ]:
# ── 5b. Low-activity flag: sessions < 5 in a month ───────────────────────────
low_activity = master[master["sessions"] < 5]
low_pct      = round(len(low_activity) / total_records * 100, 1)

print(f"Records with sessions < 5   : {len(low_activity):,}")
print(f"Share of all records        : {low_pct}%")
print()

# Unique customers with at least one low-activity month
low_customers = low_activity["customer_id"].nunique()
print(f"Distinct customers with ≥1 low-activity month : {low_customers}")

In [ ]:
# ── 5c. Feature usage score distribution by band ─────────────────────────────
bins   = [0, 3, 5, 7, 10]
labels = ["Low (0–3)", "Moderate (3–5)", "Good (5–7)", "High (7–10)"]

master["feature_band"] = pd.cut(master["feature_usage"], bins=bins, labels=labels, include_lowest=True)

band_counts = master["feature_band"].value_counts().reindex(labels, fill_value=0)
band_pct    = (band_counts / total_records * 100).round(1)

band_summary = pd.DataFrame({
    "Record Count": band_counts,
    "Share (%)": band_pct
})

print("=== Feature Usage Score Distribution ===")
print(band_summary.to_string())

**Interpretation:**  
- A high share of records with low `feature_usage` scores (0–3) indicates shallow product adoption and potential churn risk.  
- Customers with repeated `sessions < 5` months are the primary target for the early-warning intervention model.  
- The KPI target is top-2 feature usage per customer ≥ 7.0/10; the band distribution shows how far the base currently sits from that goal.

---
## 6. Churn Overview



In [ ]:
# ── 6a. Churn flag at customer level ─────────────────────────────────────────
# Unique customers in the master dataframe
unique_customers = master["customer_id"].nunique()

# Churned customers present in master
churned_in_master = master[master["is_churned"] == 1]["customer_id"].nunique()

churn_rate = round(churned_in_master / unique_customers * 100, 1)

print(f"Total unique customers       : {unique_customers}")
print(f"Churned customers            : {churned_in_master}")
print(f"Active (non-churned)         : {unique_customers - churned_in_master}")
print(f"Overall customer churn rate  : {churn_rate}%")

In [ ]:
# ── 6b. Churn reasons breakdown ──────────────────────────────────────────────
reason_counts = churn["reason"].value_counts()
reason_pct    = (reason_counts / len(churn) * 100).round(1)

reason_summary = pd.DataFrame({
    "Churned Customers": reason_counts,
    "Share (%)": reason_pct
})

print("=== Churn Reasons ===")
print(reason_summary.to_string())

In [ ]:
# ── 6c. Churn by company size (high-level) ───────────────────────────────────
# Merge churn flag into customer profile
customer_churn = customer_profile.copy()
customer_churn["is_churned"] = customer_churn["customer_id"].isin(churned_ids).astype(int)

size_churn = customer_churn.groupby("company_size")["is_churned"].agg(
    Total="count",
    Churned="sum"
)
size_churn["Churn Rate (%)"] = (size_churn["Churned"] / size_churn["Total"] * 100).round(1)
size_churn = size_churn.reindex(size_order, fill_value=0)

print("=== Churn by Company Size ===")
print(size_churn.to_string())

In [ ]:
# ── 6d. Churn by industry (high-level) ───────────────────────────────────────
industry_churn = customer_churn.groupby("industry")["is_churned"].agg(
    Total="count",
    Churned="sum"
)
industry_churn["Churn Rate (%)"] = (
    industry_churn["Churned"] / industry_churn["Total"] * 100
).round(1)
industry_churn = industry_churn.sort_values("Churn Rate (%)", ascending=False)

print("=== Churn by Industry ===")
print(industry_churn.to_string())

**Interpretation:**  
- The overall churn rate provides the baseline against the KPI target of < 5%.  
- Churn reasons give the first qualitative signal — "Lack of Features" vs "Budget Cut" require very different interventions.  
- Size and industry churn rates here are directional; detailed cross-segment analysis (size × plan × industry) follows in the Churn Analysis notebook.

---
## 7. Feature Adoption



In [ ]:
# ── 7a. Top feature value counts ─────────────────────────────────────────────
feature_counts = master["top_feature"].value_counts()
feature_pct    = (feature_counts / len(master) * 100).round(1)

feature_summary = pd.DataFrame({
    "Month-Customer Records": feature_counts,
    "Share (%)": feature_pct
})

print("=== Top Feature Distribution (all customer-months) ===")
print(feature_summary.to_string())

In [ ]:
# ── 7b. Most and least used features ─────────────────────────────────────────
most_used  = feature_counts.idxmax()
least_used = feature_counts.idxmin()

print(f"Most used feature  : {most_used}  ({feature_counts[most_used]:,} records, "
      f"{feature_pct[most_used]}%)")
print(f"Least used feature : {least_used}  ({feature_counts[least_used]:,} records, "
      f"{feature_pct[least_used]}%)")

In [ ]:
# ── 7c. Feature adoption by company size ─────────────────────────────────────
# Count of records per size × top_feature
size_feature = master.groupby(["company_size", "top_feature"]).size().reset_index(name="count")

# Find the top feature for each company size
idx           = size_feature.groupby("company_size")["count"].idxmax()
top_by_size   = size_feature.loc[idx].set_index("company_size")[["top_feature", "count"]]
top_by_size   = top_by_size.reindex(size_order)
top_by_size.columns = ["Top Feature", "Record Count"]

print("=== Most Common Top Feature by Company Size ===")
print(top_by_size.to_string())

**Interpretation:**  
- Features appearing most frequently as `top_feature` are the product's stickiest capabilities — they should be protected in any pricing restructure.  
- Features with very low adoption may indicate discoverability problems or genuine lack of fit.  
- The Churn Analysis notebook will test whether customers whose `top_feature` is "AI Insights" or "Automation" have meaningfully lower churn rates (a key hypothesis from the problem statement).

---
## 8. Basic Correlation Analysis



In [ ]:
# ── 8a. Correlation matrix ───────────────────────────────────────────────────
numeric_master = master.select_dtypes(include="number")
corr_matrix    = numeric_master.corr().round(3)

print("=== Correlation Matrix (Numeric Fields) ===")
print(corr_matrix.to_string())

In [ ]:
# ── 8b. Top correlations with is_churned ─────────────────────────────────────
churn_corr = corr_matrix["is_churned"].drop("is_churned").sort_values(key=abs, ascending=False)

print("=== Correlations with Churn Flag (strongest first) ===")
for field, val in churn_corr.items():
    direction = "positive" if val > 0 else "negative"
    print(f"  {field:<28} {val:>7.3f}  ({direction})")

In [ ]:
# ── 8c. Top correlations with subscription_revenue ───────────────────────────
rev_corr = corr_matrix["subscription_revenue"].drop("subscription_revenue").sort_values(
    key=abs, ascending=False
)

print("=== Correlations with Subscription Revenue (strongest first) ===")
for field, val in rev_corr.items():
    direction = "positive" if val > 0 else "negative"
    print(f"  {field:<28} {val:>7.3f}  ({direction})")

**Interpretation:**  
- Strong positive correlations with `subscription_revenue` (e.g., `monthly_price`, `max_users`) are expected — they confirm data integrity.  
- Correlations between `is_churned` and usage fields (sessions, active_users, feature_usage) are the early signal for the churn early-warning model developed in the Churn Analysis notebook.  
- Negative correlation between `tenure_months` and `is_churned` would confirm that newer customers churn more — a common SaaS pattern.

---
## 9. Simple Outlier Review


In [ ]:
# ── 9a. Extract describe() for all numeric fields ────────────────────────────
desc = master.select_dtypes(include="number").describe().round(2)
print(desc.to_string())

In [ ]:
# ── 9b. Flag fields where max > 3× the 75th percentile ──────────────────────
print("=== Outlier Flags (max > 3 × Q3) ===")
flagged = []
for col in desc.columns:
    q3  = desc.loc["75%", col]
    mx  = desc.loc["max", col]
    mn  = desc.loc["min", col]

    if q3 > 0 and mx > 3 * q3:
        flagged.append(col)
        print(f"  {col}")
        print(f"    Q3 = {q3},  Max = {mx},  Ratio = {round(mx / q3, 1)}×")

    if mn < 0:
        print(f"  {col}  ← NEGATIVE minimum detected (min = {mn})")

if not flagged:
    print("  No extreme outliers detected by this rule.")

In [ ]:
# ── 9c. Customers with unusually high expansion revenue in any single month ────
high_exp_threshold = master["expansion_revenue"].quantile(0.99)
high_exp_records   = master[master["expansion_revenue"] > high_exp_threshold]

print(f"99th percentile expansion_revenue : \${high_exp_threshold:.2f}")
print(f"Records above 99th percentile     : {len(high_exp_records)}")
print()
print("Top 10 expansion revenue months:")
print(
    high_exp_records[["customer_id", "month", "expansion_revenue", "plan_name"]]
    .sort_values("expansion_revenue", ascending=False)
    .head(10)
    .to_string(index=False)
)

**Interpretation:**  
- Expansion revenue is the most likely field to contain skewed outliers — a small number of upsell events can dwarf the typical value.  
- Negative values in discount or revenue fields would indicate data entry errors needing correction upstream.  
- Outliers in `sessions` or `login_count` may represent power users or automated scripts rather than true human engagement.

---
## 10. EDA Summary — Key Observations

The following observations emerge from the exploratory analysis. Each one informs a specific follow-on analysis or business question.

---

### 10.1 Customer Composition
- **500 customers** across 8 industries and 8 countries, with data spanning January 2021 – June 2024.
- Industry mix is relatively balanced, but **EdTech** is flagged as a high-churn segment requiring dedicated investigation.
- **Small companies** represent the largest size segment. Their plan alignment is a critical pricing question.
- **USA and India** are the dominant markets, shaping where pricing and support investments will have the most impact.

---

### 10.2 Plan Adoption
- All four tiers (Basic, Pro, Business, Enterprise) are represented.
- The concentration of customers on lower-priced plans limits the theoretical MRR ceiling.
- A meaningful number of Small customers on Business/Enterprise plans signals a potential **plan mismatch** — this will be quantified in the Churn Analysis notebook.

---

### 10.3 Revenue Distributions
- The gap between mean `base_revenue` and mean `subscription_revenue` confirms that **discounting is systematically reducing realized revenue** across all plans.
- **Expansion revenue is a minority event** — it appears in well under 50% of customer-months, confirming the upsell gap identified in the problem statement.
- Revenue distributions are right-skewed: a small number of Enterprise customers drive outsized values.

---

### 10.4 Usage Distributions
- A non-trivial share of customer-months have **very low session counts (< 5)**, indicating a segment of at-risk customers.
- `feature_usage` scores cluster in the low-to-moderate range for many records, suggesting broad shallow adoption rather than deep engagement with any single feature.
- The KPI target of feature usage ≥ 7.0/10 for top-2 features appears achievable for a minority of the base in the current data.

---

### 10.5 Churn Overview
- **36 customers have churned** out of 500, giving a raw customer churn rate of **7.2%** — above the < 5% KPI target.
- The most common churn reasons ("Lack of Features", "Budget Cut", "Poor Support") point to product-market fit and pricing sensitivity as the primary levers.
- **EdTech** and specific size segments show disproportionately elevated churn rates at first glance — to be confirmed with full cross-segment analysis.

---

### 10.6 Feature Adoption
- Product usage is concentrated in a small number of features. The most-used feature attracts a majority of top-feature records.
- **AI Insights and Automation** — the two features hypothesized to be retention drivers — will be tested against churn rates in the Churn Analysis notebook.
- Feature adoption varies by company size, which may inform plan-feature bundling decisions in the Pricing notebook.

---

### 10.7 Correlations and Outliers
- Usage metrics (sessions, active_users, feature_usage) show the expected directional correlation with churn — lower engagement associates with higher churn probability.
- `tenure_months` shows a negative correlation with `is_churned`, consistent with the well-established SaaS pattern that newer customers are at higher churn risk.
- Expansion revenue contains right-tail outliers driven by a small set of large upsell events, which will require careful treatment in revenue trend analysis.

---

